# GitHub and Kubernetes incident evidence
Inspect repository and cluster API activity with synthetic records, schema-validated OCSF and a zero-call AI plan. Audit events describe recorded actions; they do not establish malicious intent. This notebook makes no provider requests.

In [ ]:
from pathlib import Path
import json
import tempfile
from timeline_demo.pipeline import Input, run_pipeline, read_timeline
from timeline_demo.ocsf import export_bundle, verify_export
from timeline_demo.ai import prepare
from timeline_demo.operations import inventory
root = Path.cwd()
if not (root/'examples').exists():
    root = root.parent
samples = json.loads((root/'examples/parser_samples.json').read_text())
work = tempfile.TemporaryDirectory()
workdir = Path(work.name)
inputs = []
for parser in ['github_audit', 'kubernetes_audit']:
    path = workdir/(parser + '.json')
    path.write_text(json.dumps(samples[parser]))
    inputs.append(Input(parser, path))
bundle = workdir/'bundle'
manifest = run_pipeline(inputs, bundle, 'developer-incident-demo')
events = list(read_timeline(bundle))
assert manifest['counts']['event_count'] == 2
[(event['product_name'], event['activity_name'], event['status']) for event in events]

In [ ]:
report = export_bundle(bundle, workdir/'ocsf')
assert verify_export(workdir/'ocsf', bundle=bundle) == report
plan = prepare(bundle, 'summarize')
print({'model': plan['model'], 'coverage': plan['coverage'], 'model_calls': 0})
configs = [json.loads((root/'examples/collectors'/name).read_text()) for name in
           ['github-audit.json', 'cloudwatch-kubernetes.json']]
inventory(configs)

Use the generated inventory with `timeline-ops health --inventory FILE` on collected state to detect sources that never started. GitHub collection is organization-scoped on api.github.com; Kubernetes historical audits must come from an enabled log backend or exported files. Different audit stages share an auditID but retain distinct evidence identities. The original authenticated user and any impersonated user remain separate in raw evidence. Review docs/DEVELOPER_INCIDENTS.md before tenant acceptance.

In [ ]:
work.cleanup()